# Data Preparation (Preparação dos dados)

### Biblioteca / Configuração

In [44]:
# Dependências
import sys
#!{sys.executable} -m pip install --disable-pip-version-check -r ../requirements.txt -q
print('Bibliotecas instaladas')

Bibliotecas instaladas


In [45]:
# Acesso aos modulos do diretório
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT)) 

# Manipulação dos dados
import pandas as pd
import numpy as np
import pickle

# Visualização dos dados
import matplotlib.pyplot as plt
import seaborn as sns

# Diretórios
from config.paths import *
from config.data_preprocessing import *

# Avisos
import warnings
warnings.filterwarnings('ignore')

# Configuração
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', None)

print('Ambiente Configurado')

Ambiente Configurado


## Parâmetros Globais

In [46]:
# define a coluna alvo do modelo
TARGET = 'FPD'
# percentual máximo de valores ausentes permitido para manter a variável
PERCENTUAL_MAX_FALTANTES = 70

### Carregamento dos dados 

In [47]:
# lê o arquivo parquet e carrega em um DataFrame
abt00 = pd.read_parquet(RAW_DIR / 'book_variaveis_04.parquet')

In [48]:
abt00.drop(columns=['var_24'], inplace=True)

## Tratamento inicial

### Grupo Controle

In [49]:
# gerar e salvar o grupo controle no data/processed

# cria flag para identificar clientes do grupo controle (CPF 6º e 7º dígitos = ZZ ou ZX)
abt00['FLAG_GRUPO_CONTROLE'] = (abt00['NUM_CPF'].astype(str).str[5:7].isin(['ZZ', 'ZX']).astype(int))

# filtra grupo controle
controle = abt00[abt00['FLAG_GRUPO_CONTROLE'] == 1]

# salva em parquet usando path centralizado
GRUPO_CONTROLE_FILE = RAW_DIR / 'grupo_controle.parquet'
controle.to_parquet(GRUPO_CONTROLE_FILE, index=False)

print(f"Grupo controle salvo em: {GRUPO_CONTROLE_FILE}")
print(f"Registros: {len(controle):,}")

Grupo controle salvo em: C:\pod\hackathon_pod_2025\ciencia\01_data\raw\grupo_controle.parquet
Registros: 60,197


### Definir filtro grupo controle

In [50]:
# controla aplicação do filtro e remove coluna se só houver grupo controle
APLICAR_FILTRO_PADRAO = True  # True = sem grupo controle | False = base completa

if APLICAR_FILTRO_PADRAO:
    abt01 = abt00[abt00['FLAG_GRUPO_CONTROLE'] == 0].copy()

    # se depois do filtro só existir grupo controle, remove a coluna
    if 'FLAG_GRUPO_CONTROLE' in abt01.columns and abt01['FLAG_GRUPO_CONTROLE'].nunique() == 1:
        abt01.drop(columns=['FLAG_GRUPO_CONTROLE'], inplace=True)
else:
    abt01 = abt00.copy()

print(f"Modo ativo: {'Sem grupo controle' if APLICAR_FILTRO_PADRAO else 'base completa'}")
print(f"Base ativa: {len(abt01):,} registros")

Modo ativo: Sem grupo controle
Base ativa: 1,220,631 registros


### Separação dos dados para validação temporal (Out-of-Time)

A separação dos dados é realizada com base na **safra**, respeitando a ordem temporal das observações.  
Essa abordagem, conhecida como **validação Out-of-Time (OOT)**, evita vazamento de informação e simula o comportamento real do modelo em dados futuros.

In [51]:
# garante SAFRA como inteiro
abt01['SAFRA'] = abt01['SAFRA'].astype(int)
safra_counts = abt01['SAFRA'].value_counts().sort_index()

# Definir SAFRAs de teste (Fevereiro e Março 2025)
test_safras = [202502, 202503]

# Criar máscaras
test_mask = abt01['SAFRA'].isin(test_safras)
train_mask = ~test_mask

# Separar dados
train = abt01[train_mask].copy()
test = abt01[test_mask].copy()

train.shape, test.shape

((831081, 128), (389550, 128))

In [52]:
# salvar base de teste em parquet no diretório de predictions
TEST_FILE = RAW_DIR / 'base_test.parquet'
test.to_parquet(TEST_FILE, index=False)

print(f"Base de teste salva em: {TEST_FILE}")
print(f"Registros: {len(test):,}")

Base de teste salva em: C:\pod\hackathon_pod_2025\ciencia\01_data\raw\base_test.parquet
Registros: 389,550


In [53]:
# Backup dos dados originais
train_01 = train.copy()

# lista de vars para retirar dos tratamentos e algumas colunas desnecessárias
ignore_cols = ['SAFRA', 'FPD', 'NUM_CPF', 'DATADENASCIMENTO', 'REGIAO_POSTAL_TXT', 'var_25']

# Aplicando no treino
train_01 = train_01.drop(columns=ignore_cols)

>Motivo da exclusão

- `REGIAO_POSTAL_TXT` replica o conteúdo de `REGIAO_POSTAL`, diferenciando-se apenas pelo tipo de dado.
- `var25` (categórica original) foi removida após sua binarização em variáveis dummies, evitando redundância e multicolinearidade no modelo.

In [54]:
# exibir header claro para geração da tabela de metadados do treino

print("METADADOS DO DATASET DE TREINO")
print('=' * 30)
print(f"Linhas: {train_01.shape[0]:,} | Colunas: {train_01.shape[1]:,}")
print('Gerando tabela de diagnóstico das variáveis...\n')

metadados = dataset_info_table(train_01)

print('OK: tabela de metadados criada com sucesso.')

METADADOS DO DATASET DE TREINO
Linhas: 831,081 | Colunas: 122
Gerando tabela de diagnóstico das variáveis...

OK: tabela de metadados criada com sucesso.


In [55]:
metadados

,Feature,QT_nulos,PC_nulos,QT_zeros,Cardinalidade,Tipo_feature
0,VALOR_SOS_ULT_6_SAFRAS,480525,57.82,350556,1,float64
1,RATIO_BONUS_CREDITO,480525,57.82,336454,78,float64
2,QTDE_RECARGAS_ULT_3_SAFRAS,480525,57.82,17294,25,float64
3,QTDE_RECARGAS_ULT_1_SAFRAS,480525,57.82,17294,9,float64
4,FLAG_SEM_RECARGA,480525,57.82,350556,1,float64
...,...,...,...,...,...,...
117,var_91,0,0.00,0,15,int64
118,var_90,0,0.00,69,1223,float64
119,var_89,0,0.00,83297,106,int64
120,var_88,0,0.00,0,13,int64


### Remoção de Colunas Desnecessárias

In [56]:
# filtra variáveis com muitos nulos e cardinalidade igual a 1
df_low_card = metadados[(metadados['PC_nulos'] >= PERCENTUAL_MAX_FALTANTES) | (metadados['Cardinalidade'] <= 1)]
df_low_card = list(df_low_card.Feature.values)

# efeito real do drop
qtd_excluir = train_01.columns.isin(df_low_card).sum()

print(qtd_excluir)
print(df_low_card)

17
['VALOR_SOS_ULT_6_SAFRAS', 'FLAG_SEM_RECARGA', 'FLAG_TEVE_ESTORNO', 'TICKET_MEDIO_SOS', 'RATIO_SOS_RECARGAS', 'FLAG_ESTORNO_REAL', 'FLAG_ESTORNO_BONUS', 'VALOR_SOS', 'QTD_SOS', 'QTD_SOS_ULT_1_SAFRAS', 'VALOR_SOS_ULT_3_SAFRAS', 'VALOR_SOS_ULT_1_SAFRAS', 'QTD_SOS_ULT_6_SAFRAS', 'QTD_SOS_ULT_3_SAFRAS', 'flag_mig2', 'PROD', 'FLAG_INSTALACAO']


In [57]:
# filtra variáveis com cardinalidade maior igual a 100000
df_high_card = metadados[(metadados['Cardinalidade'] >= 100000)]
df_high_card = list(df_high_card.Feature.values)

# efeito real do drop
qtd_excluir = train_01.columns.isin(df_high_card).sum()

print(qtd_excluir)
print(df_high_card)

0
[]


In [58]:
# Unir colunas de baixa e alta cardinalidade
drop_card = list(set(df_low_card + df_high_card))
# Remover colunas de baixa e alta cardinalidade
train_01 = train_01.drop(columns=drop_card, errors='ignore')

In [59]:
# colunas que permaneceram após o drop
features_pre_selection = train_01.columns.tolist()

artifact_path = Path(ARTIFACT_DIR) / 'features_pre_selection.pkl'

with open(artifact_path, 'wb') as f:
    pickle.dump(features_pre_selection, f)

print(f"Features pre seleção: {len(features_pre_selection)}")
print(f"Arquivo: {artifact_path}")

Features pre seleção: 105
Arquivo: C:\pod\hackathon_pod_2025\ciencia\04_artifact\features_pre_selection.pkl


### Tratamento de Valores Faltantes

In [60]:
# Seleciona apenas colunas do tipo object/categoricas
cols_obj = train_01.select_dtypes(include='object').columns

# Identifica colunas que têm algum valor parecido com "desconhecido" (qualquer capitalização ou espaços)
cols_com_desconhecido = [
    col for col in cols_obj
    if train_01[col].astype(str).str.strip().str.match(r'(?i)^desconhecido$').any()
]
# Substituir todas as variações de "Desconhecido" por NaN
for col in cols_com_desconhecido:
    train_01[col] = train_01[col].astype(str).str.strip().replace(r'(?i)^desconhecido$', np.nan, regex=True)

In [61]:
# Identifica colunas object que sobraram (categóricas de verdade)
cols_categoricas = train_01.select_dtypes(include='object').columns.tolist()
print("Colunas categóricas restantes:", cols_categoricas)

# Verifica cardinalidade de cada uma pra decidir o tratamento
for col in cols_categoricas:
    print(f"\n{col} | cardinalidade: {train_01[col].nunique()}")
    print(train_01[col].value_counts(dropna=False).head(5))

Colunas categóricas restantes: ['var_03', 'var_04', 'var_05', 'var_09', 'REGIAO_POSTAL', 'SUB_REGIAO_POSTAL']

var_03 | cardinalidade: 100
33      253002
1        56930
None     53539
3        36352
50       23139
Name: var_03, dtype: int64

var_04 | cardinalidade: 6
0    758244
1     44579
2     17404
3      6470
4      2449
Name: var_04, dtype: int64

var_05 | cardinalidade: 10
1       386039
2       304361
3        55635
None     32345
4        28821
Name: var_05, dtype: int64

var_09 | cardinalidade: 15
None    442366
9       229909
8        53071
5        34926
7        31360
Name: var_09, dtype: int64

REGIAO_POSTAL | cardinalidade: 10
6    122094
7    105076
1     96940
0     93747
2     91359
Name: REGIAO_POSTAL, dtype: int64

SUB_REGIAO_POSTAL | cardinalidade: 99
NaN    45936
13     31869
69     27113
65     23036
55     19371
Name: SUB_REGIAO_POSTAL, dtype: int64


In [62]:

cols_nao_numericas = train_01.select_dtypes(exclude=['float', 'int']).columns.tolist()

for col in cols_nao_numericas:
    # Tenta converter pra numérico; o que não der vira NaN
    convertida = pd.to_numeric(train_01[col], errors='coerce')
    
    # Só substitui a coluna se pelo menos 1 valor converteu com sucesso
    if convertida.notna().any():
        train_01[col] = convertida.astype('Int64')
    else:
        print(f"Pulando '{col}' — coluna de texto (ex: '{train_01[col].dropna().iloc[0]}')")

In [63]:
# Análise de missing values restantes
train_01, stats = custom_fillna(train_01)

In [64]:
# salva estatísticas de imputação
artifact_path = Path(ARTIFACT_DIR) / 'stats_nulo.pkl'
with open(artifact_path, "wb") as f:
    pickle.dump(stats, f)

print(f"Colunas com imputação aplicada: {len(stats)}")
print(f"Artefato salvo em: {artifact_path}")

Colunas com imputação aplicada: 3
Artefato salvo em: C:\pod\hackathon_pod_2025\ciencia\04_artifact\stats_nulo.pkl


## Salvamento dos Dados Processados

In [65]:
# Reanexar target Para treino
abt01_train_final = train_01.copy()
abt01_train_final[TARGET] = train[TARGET].values

In [66]:
# salvar base de Treino tratada
TRAIN_FILE = PROCESSED_DIR / 'abt01_train.parquet'
print('\n💾 Salvando dados processados...')

# salvar dataset final
abt01_train_final.to_parquet(TRAIN_FILE, index=False)
print(f"✓ Treino salvo: {TRAIN_FILE}")
print('✅ Persistência concluída')


💾 Salvando dados processados...
✓ Treino salvo: C:\pod\hackathon_pod_2025\ciencia\01_data\processed\abt01_train.parquet
✅ Persistência concluída
